In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MultiLabelBinarizer
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Input
from tensorflow.keras.optimizers import Adam

# Load data
file_path = "C:/Data/restaurant_preferences.xlsx"
data = pd.read_excel(file_path)

usernames = data.iloc[:, 0]
restaurants = data.iloc[:, 1:].fillna("")

restaurant_lists = restaurants.apply(lambda x: [r for r in x if r != ""], axis=1)

mlb = MultiLabelBinarizer()
one_hot_restaurants = mlb.fit_transform(restaurant_lists)
restaurant_df = pd.DataFrame(one_hot_restaurants, columns=mlb.classes_)

X, y = [], []
for row in restaurant_df.itertuples(index=False):
    liked_restaurants = np.where(row)[0]
    if len(liked_restaurants) < 2:
        continue
    for left_out_idx in liked_restaurants:
        input_features = np.array(row, dtype=int)
        input_features[left_out_idx] = 0
        X.append(input_features)
        y.append(left_out_idx)

X, y = np.array(X), np.array(y)

# Use Input() layer instead of deprecated input_shape argument on Dense
model = Sequential([
    Input(shape=(X.shape[1],)),
    Dense(16, activation='relu'),
    Dense(8, activation='relu'),
    Dense(X.shape[1], activation='softmax'),
])

model.compile(optimizer=Adam(learning_rate=0.001),
              loss='sparse_categorical_crossentropy', metrics=['accuracy'])

model.fit(X, y, epochs=10, batch_size=32, validation_split=0.2)

def recommend_restaurant(selected_restaurants):
    input_vector = np.zeros(X.shape[1])
    for restaurant in selected_restaurants:
        if restaurant in mlb.classes_:
            input_vector[mlb.classes_.tolist().index(restaurant)] = 1
    prediction = model.predict(input_vector.reshape(1, -1), verbose=0)
    recommended_index = np.argmax(prediction)
    return mlb.classes_[recommended_index]

In [7]:
import tkinter as tk
from tkinter import messagebox
from tensorflow.keras.models import load_model  # If you saved your model, you can load it

# Initialize main GUI window
root = tk.Tk()
root.title("Restaurant Recommender System")
root.geometry("400x300")

# Load the unique restaurant list for dropdown options
restaurant_list = mlb.classes_.tolist()

# Variables to store selected restaurants
selected_restaurants = [tk.StringVar() for _ in range(4)]

# Create dropdowns for restaurant selection
dropdown_labels = ["Restaurant 1", "Restaurant 2", "Restaurant 3", "Restaurant 4"]
for i in range(4):
    tk.Label(root, text=dropdown_labels[i]).pack()
    tk.OptionMenu(root, selected_restaurants[i], *restaurant_list).pack()

# Function to get recommendations based on selected restaurants
def get_recommendation():
    # Gather selected restaurant choices
    selected = [var.get() for var in selected_restaurants]
    
    # Check if all dropdowns have a valid selection
    if "" in selected:
        messagebox.showwarning("Incomplete Selection", "Please select four restaurants.")
        return
    
    # Use the model to predict the recommended restaurant
    recommended_restaurant = recommend_restaurant(selected)
    messagebox.showinfo("Recommendation", f"We recommend you try: {recommended_restaurant}")

# Button to get recommendations
tk.Button(root, text="Get Recommendation", command=get_recommendation).pack()

# Run the GUI application
root.mainloop()

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 47ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pandas as pd
import warnings; warnings.filterwarnings('ignore')
plt.rcParams.update({'figure.dpi': 110})

restaurant_names = list(mlb.classes_)

# ── Easy: Restaurant Visit Frequency Bar ─────────────────────────────────────
visit_counts = restaurant_df.sum(axis=0).sort_values(ascending=False)
fig, ax = plt.subplots(figsize=(14, 6))
colors_bar = plt.cm.viridis(np.linspace(0.9, 0.2, len(visit_counts)))
bars = ax.bar(visit_counts.index, visit_counts.values, color=colors_bar, edgecolor='white', linewidth=0.8)
ax.set_title('Restaurant Popularity — Number of Users Who Visited', fontsize=14, fontweight='bold')
ax.set_ylabel('Visit Count'); ax.tick_params(axis='x', rotation=60, labelsize=8)
ax.spines[['top','right']].set_visible(False)
for bar in bars:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1,
            f'{int(bar.get_height())}', ha='center', fontsize=7)
plt.tight_layout(); plt.show()

# ── Medium: Restaurant Co-occurrence Heatmap ─────────────────────────────────
cooccur = restaurant_df.T @ restaurant_df   # (n_rest, n_rest)
np.fill_diagonal(cooccur.values, 0)         # remove self-counts

fig, ax = plt.subplots(figsize=(14, 11))
mask_zero = cooccur == 0
sns.heatmap(cooccur, ax=ax, cmap='YlOrRd', mask=mask_zero,
            linewidths=0.4, linecolor='white',
            xticklabels=True, yticklabels=True,
            cbar_kws={'label': 'Co-visit count', 'shrink': 0.6})
ax.set_title('Restaurant Co-occurrence Matrix\n(how often pairs of restaurants are visited by the same person)',
             fontsize=13, fontweight='bold')
ax.tick_params(axis='x', rotation=60, labelsize=7)
ax.tick_params(axis='y', rotation=0,  labelsize=7)
plt.tight_layout(); plt.show()

# ── Medium: Model Prediction Confidence Distribution ─────────────────────────
probs = model.predict(X, verbose=0)
top1_conf   = probs.max(axis=1)
correct_mask = (probs.argmax(axis=1) == y)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
axes[0].hist(top1_conf[correct_mask],   bins=30, alpha=0.7, color='#2ecc71',
             density=True, label=f'Correct ({correct_mask.sum()})')
axes[0].hist(top1_conf[~correct_mask],  bins=30, alpha=0.7, color='#e74c3c',
             density=True, label=f'Wrong ({(~correct_mask).sum()})')
axes[0].set_title('Prediction Confidence: Correct vs Wrong', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Max Softmax Probability'); axes[0].set_ylabel('Density')
axes[0].legend(fontsize=11)

# Top-5 accuracy
top5_correct = np.sum([y[i] in np.argsort(probs[i])[-5:] for i in range(len(y))])
top1_acc = correct_mask.mean()
axes[1].bar(['Top-1 Accuracy', 'Top-5 Accuracy'],
            [top1_acc, top5_correct/len(y)],
            color=['#3498db','#8e44ad'], width=0.45, edgecolor='white', linewidth=2)
axes[1].set_ylim(0, 1); axes[1].set_ylabel('Accuracy')
axes[1].set_title('Top-1 vs Top-5 Recommendation Accuracy', fontsize=13, fontweight='bold')
for i, v in enumerate([top1_acc, top5_correct/len(y)]):
    axes[1].text(i, v + 0.02, f'{v:.1%}', ha='center', fontsize=13, fontweight='bold')
axes[1].spines[['top','right']].set_visible(False)
plt.tight_layout(); plt.show()

# ── Hard: t-SNE of User Preference Vectors ───────────────────────────────────
from sklearn.manifold import TSNE
# Each user's preference vector = sum of one-hot restaurant vectors
user_vectors = restaurant_df.values  # (n_users, n_restaurants)
n_users_unique = len(usernames.unique())

if len(user_vectors) > 10:
    tsne_perp = min(30, len(user_vectors) - 1)
    X_2d_rest = TSNE(n_components=2, random_state=42, perplexity=tsne_perp,
                     learning_rate='auto', init='pca').fit_transform(user_vectors)
    total_visits = user_vectors.sum(axis=1)

    fig, ax = plt.subplots(figsize=(12, 9))
    scatter = ax.scatter(X_2d_rest[:,0], X_2d_rest[:,1], c=total_visits,
                         cmap='plasma', s=60, alpha=0.75)
    cb = plt.colorbar(scatter, ax=ax)
    cb.set_label('Total Restaurants Visited', fontsize=11)
    for i, name in enumerate(usernames):
        ax.annotate(str(name), (X_2d_rest[i,0], X_2d_rest[i,1]),
                    fontsize=6, alpha=0.6, xytext=(2,2), textcoords='offset points')
    ax.set_title('t-SNE of User Restaurant Preference Vectors\n(proximity = similar taste profiles)',
                 fontsize=13, fontweight='bold')
    ax.set_xlabel('t-SNE 1'); ax.set_ylabel('t-SNE 2')
    plt.tight_layout(); plt.show()

# ── Hard: Recommendation Output Probability Heatmap for Top Users ────────────
# Show predicted probabilities for first 15 users across all restaurants
n_show = min(15, len(X))
sample_probs = model.predict(X[:n_show], verbose=0)
fig, ax = plt.subplots(figsize=(16, 8))
sns.heatmap(sample_probs, ax=ax, cmap='YlOrRd',
            xticklabels=restaurant_names, yticklabels=range(1, n_show+1),
            linewidths=0.3, linecolor='white',
            cbar_kws={'label': 'Predicted Probability', 'shrink': 0.6})
ax.set_title(f'Recommendation Probability Heatmap — First {n_show} Training Samples\n(rows = samples, cols = restaurants)',
             fontsize=13, fontweight='bold')
ax.set_xlabel('Restaurant'); ax.set_ylabel('Sample Index')
ax.tick_params(axis='x', rotation=60, labelsize=8)
plt.tight_layout(); plt.show()